---
title: NBA Team Performance-Based Rewards
project: NBA Long-Term Performance Rewards
file_type: notebook
status: active
purpose: Generate and inspect long-term NBA reward and penalty reports.
usage: Run top to bottom after Data Collection has published the required historical inputs.
last_updated: 2026-08-16
---

# NBA Playoffs and Champions Analysis

This notebook is the interactive entry point for the same report workflow used by `Scripts/run_pipeline.py`. The analytical logic remains in `Scripts/generate_reports.py`, preventing the notebook and command-line pipeline from producing different results.

Run the cells from top to bottom. Each execution creates a new immutable timestamped folder under `Reports/`; it never overwrites an earlier run.

## 1. Initialize the project

The setup cell locates the project root whether Jupyter opened in the project root or the `Notebooks` folder, then imports the shared production functions.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

# Find the project by its stable active folders instead of assuming Jupyter's start location.
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (path for path in search_locations if (path / "Scripts" / "generate_reports.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the NBA Long-Term Performance Rewards project. "
        "Start Jupyter from the project root or its Notebooks folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Scripts.common import configured_path, load_config
from Scripts.generate_reports import generate_reports

## 2. Choose and review run settings

Leave `run_name` as `None` for the standard timestamp-only folder, or provide a short label such as `"baseline"`. To use another governed configuration, change `config_path`. This cell displays the effective source paths and thresholds before any reports are generated.

In [ ]:
config_path = PROJECT_ROOT / "Config" / "analysis_settings.json"
run_name = None

config = load_config(config_path)
input_review = pd.DataFrame(
    [
        {"Input": key, "Resolved path": str(configured_path(config, key))}
        for key in ("standings", "playoffs", "team_mapping")
    ]
)
threshold_review = pd.DataFrame(
    [{"Threshold": key, "Value": value} for key, value in config["thresholds"].items()]
)

display(Markdown(f"**Configuration:** `{config_path}`  \n**Run label:** `{run_name}`"))
display(input_review)
display(threshold_review)

## 3. Generate reports

This calls the same `generate_reports` function used by the pipeline script. A successful run updates `Reports/latest_run.json`; an exception is allowed to remain visible so incomplete or invalid runs are not mistaken for completed output.

In [ ]:
reports = generate_reports(config, run_name=run_name)

reports_root = configured_path(config, "reports_dir")
latest_pointer = json.loads((reports_root / "latest_run.json").read_text(encoding="utf-8"))
run_directory = reports_root / latest_pointer["run_directory"]
report_summary = pd.DataFrame(
    [
        {"Report": report_name, "Rows": len(report_df)}
        for report_name, report_df in reports.items()
    ]
).sort_values("Report", ignore_index=True)

display(Markdown(
    f"### Run complete\n\n"
    f"- Run ID: `{latest_pointer['run_id']}`\n"
    f"- Output folder: `{run_directory}`\n"
    f"- Manifest: `{reports_root / latest_pointer['manifest']}`"
))
display(report_summary)

## 4. Optionally inspect a report

Choose any name shown in `report_summary`. This preview reads the in-memory result from the completed run and does not create another output folder.

In [ ]:
report_name = "Output_Playoff_Streaks.csv"

if report_name not in reports:
    raise KeyError(f"Unknown report: {report_name}. Choose a name from report_summary.")
display(reports[report_name])